# Install required dependencies


In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-huggingface langchain-text-splitters langchain-classic
!pip install -q faiss-gpu bitsandbytes accelerate wandb



# Import Required Libs

In [1]:
# Standard packages

# EDA
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import re
import random
from dataclasses import dataclass
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Pytorch and HF
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    AutoModelForMultipleChoice, 
    Trainer, 
    TrainingArguments, 
    BitsAndBytesConfig, 
    pipeline,
    DataCollatorForMultipleChoice
)

# LangChain & RAG
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.chains import RetrievalQA

print("langchain:", langchain.__version__)
print("langchain_core:", langchain_core.__version__)
print("langchain_community:", langchain_community.__version__)

# Tracking
import wandb

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1

/tmp/ipykernel_58/2698707313.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


langchain: 1.2.15
langchain_core: 1.4.9
langchain_community: 0.4.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.1 MB/s eta 0:00:00:00:0100:01
/kaggle/input/datasets/kalyandath/wiki-dataset/Wiki_knowledge_raw_gem.csv
/kaggle/input/datasets/kalyandath/wiki-dataset/Wiki_knowledge_raw.csv
/kaggle/input/datasets/kalyandath/science-dataset/physics_pages.csv
/kaggle/input/datasets/kalyandath/deberta-pretrained/config.json
/kaggle/input/datasets/kalyandath/deberta-pretrained/training_args.bin
/kaggle/input/datasets/kalyandath/deberta-pretrained/tokenizer.json
/kaggle/input/datasets/kalyandath/deberta-pretrained/tokenizer_config.json
/kaggle/input/datasets/kalyandath/deberta-pretrained/model.safetensors
/kaggle/input/datasets/kalyandath/faiss-index/index.faiss
/kaggle/input/datasets/kalyandath/faiss-index/index.pkl
/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/

# Run Config

In [2]:
EDA = False
RAG_KnowlegeBaseCreation = False
RAG_PIPELINE = False
DEBERTA_TRAINING = False
DEBERTA_INFERENCE = False
LSTM_MODEL = True

# Load the CSV files

In [3]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

---

# Exploratory Data Analysis

## Look into the data

In [4]:
if EDA:
    train_df.info()

In [5]:
if EDA:
    display(train_df.head())

## Look into the answer column

In [6]:
if EDA:
    print(f"Possible answer options in the trainig data: {sorted(train_df["answer"].unique())}")

In [7]:
if EDA:
    print("Distribution of answer options in the train data:")
    train_df["answer"].value_counts()

In [8]:
# Visualization of the Distribution
if EDA:
    train_df["answer"].value_counts().plot(kind="bar")

## Looking into the prompt column

In [9]:
if EDA:
    # Number of characters in the prompt column
    total_char = train_df["prompt"].apply(len).sum()
    total_prompts = len(train_df)
    
    print("Sum of characters in the prompt column:", total_char)
    print("Total num of prompts:", total_prompts)
    
    print("Average Character count:", total_char/total_prompts)
    print("Maximum Character count in a single prompt:", train_df["prompt"].apply(len).max())
    print("Minimum Character count in a single prompt:", train_df["prompt"].apply(len).min())
    print("Prompt with Minimum Characters:", train_df.iloc[train_df["prompt"].apply(len).idxmin()]["prompt"])
    print("Prompt with Maximum Characters:", train_df.iloc[train_df["prompt"].apply(len).idxmax()]["prompt"])

In [10]:
# Number of words in the prompt column
if EDA:
    def to_words(prompt):
        words = prompt.split()
        return len(words)
    
    total_words = train_df["prompt"].apply(to_words).sum()
    
    print("Total Number of words in the prompt column:", total_words)
    print("Total Number of prompts:", total_prompts)
    print("-"*50)
    
    print("Average word count in the prompt column:", total_words/total_prompts)
    print("Maximum word count in a single prompt:", train_df["prompt"].apply(to_words).max())
    print("Minimum word count in a single prompt:", train_df["prompt"].apply(to_words).min())

## Look into a single row of train data

In [11]:
if EDA:
    index = random.randint(0,2000)
    print(f"Prompt:, {train_df.iloc[index]["prompt"]}\n")
    print(f"Option A: {train_df.iloc[index]["A"]}\n")
    print(f"Option B: {train_df.iloc[index]["B"]}\n")
    print(f"Option C: {train_df.iloc[index]["C"]}\n")
    print(f"Option D: {train_df.iloc[index]["D"]}\n")
    print(f"Option E: {train_df.iloc[index]["E"]}\n")
    print("Answer:", train_df.iloc[index]["answer"],":", train_df.iloc[index][train_df.iloc[index]["answer"]])

## Insight

In [12]:
if EDA:
    duplicates = train_df["prompt"].duplicated().sum()
    print(f"Train data has {duplicates} duplicate questions!!! Need to drop these...")

In [13]:
train_df = train_df.drop_duplicates(subset="prompt")
print(f"After removing duplicates:")
print(f"Duplicates left:",train_df["prompt"].duplicated().sum())
print(f"Total prompts:", len(train_df))

After removing duplicates:
Duplicates left: 0
Total prompts: 1758


---

# RAG - Knowledge Base and FAISS Index Creation

## Config


In [14]:
DATA_PATH = "/kaggle/input/datasets/kalyandath/science-dataset/physics_pages.csv"
WIKI_DATH_PATH = "/kaggle/input/datasets/kalyandath/wiki-dataset/Wiki_knowledge_raw.csv"
data = pd.read_csv(DATA_PATH)
wiki_data = pd.read_csv(WIKI_DATH_PATH)

## Inspect the data

In [15]:
if RAG_KnowlegeBaseCreation:
    display(data.head())

In [16]:
if RAG_KnowlegeBaseCreation:
    display(wiki_data.head())

## Normalization to clean the data

In [17]:
if RAG_KnowlegeBaseCreation:
    print("Fields with empty content in physics data:", data["text"].isna().sum() + (data["text"] == "").sum())
    print("Fields with empty content in wiki data:", wiki_data["content"].isna().sum() + (wiki_data["content"] == "").sum())

In [18]:
if RAG_KnowlegeBaseCreation:
    # Remove rows with content less than 50 chars
    print(f"Content with less than 50 chars:{(data["text"].str.len() < 50).sum()}")
    print(f"Content with less than 50 chars for wiki data:{(wiki_data["content"].str.len() < 50).sum()}")
    print("Remove contents having less than 50 characters.")
    data = data.loc[data["text"].str.len() > 50]
    wiki_data = wiki_data.loc[wiki_data["content"].str.len() > 50]

## Function clean the text content

In [19]:
if RAG_KnowlegeBaseCreation:
    def clean_text(text):
        # 1. Remove the wikipedia citation markers
        # [1], [23]
        text = re.sub(r'\[\d+\]\s*', "", text)
    
        # 2. Remove the section headers
        # === History ===, === References ===
        text = re.sub(r'^={2,}.*?={2,}', "", text, flags=re.MULTILINE)
    
        # 3. Remove "Source:" labels
        text = re.sub(r"Source:\s*(?=$|\n)","", text)
    
        # 4. Remove common latex commands
        # \frac, \alpha, \displaystyle
        text = re.sub(r'\\[a-zA-Z]+', " ", text)
    
        # 5. Remove leftover latex symbols
        text = re.sub(r"[{}\\^~]", " ", text)
    
        # 6. Remove page-number styes
        # : 327 , :  469-470
        text = re.sub(r":\s*\d+(?:-\d+)?\b", "", text)
    
        # 7 . Normalize whitespaces
        text = "\n".join(line.strip() for line in text.splitlines())
    
        # 8. Collapse multiple blank lines
        text = re.sub(r"\n{2,}", "\n", text)
    
        # 9. Convert everything into single para.
        text = text.replace("\n"," ").replace("\r", " ")
    
        # 10. Collapse sequence of whtiespace into a single space
        text = re.sub(r"\s+", " ",text)
    
        return text.strip()

## Process and clean content, create document for chunking

In [20]:
if RAG_KnowlegeBaseCreation:
# Define the text splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap = 300,
        separators = [". ", "? ", "! ", ", ", " ", ""]
    )
    
    print("Creating the document data structure..")
    documents = []
    for index, row in data.iterrows():
        page = str(row["page"])
        title = str(row["section_title"])
        raw_text = str(row["text"])
        text = clean_text(raw_text)
        context_string = f"{page}:{title}\n\n{text}"
    
        documents.append(Document(
            page_content = context_string,
            metadata = {"source": f'{page}:{title}'}
        ))
    
    print("Successfully processed documents from physics dataset:", len(documents))
    
    for index, row in wiki_data.iterrows():
        title = str(row["title"])
        raw_text = str(row["content"])
        text = clean_text(raw_text)
        context_string = f"{title}\n\n{text}"
    
        documents.append(Document(
            page_content = context_string,
            metadata = {'source': title}
        ))
    print("Successfully processed documents from wiki dataset. Total Documents:", len(documents))

## Chunkig Documents


In [21]:
if RAG_KnowlegeBaseCreation:
    filtered_documents = [doc for doc in documents if len(doc.page_content) > 100]
    chunks = text_splitter.split_documents(filtered_documents)
    print(f"Splitted documents into {len(chunks)} chunks...")

## Create FAISS Index with the chunks

In [22]:
if RAG_KnowlegeBaseCreation:
    embeddings = HuggingFaceEmbeddings(
        model_name = "BAAI/bge-large-en-v1.5",
        model_kwargs = {'device':'cuda'}
    )
    
    # vector_store = FAISS.from_documents(
    #     documents = chunks,
    #     embedding = embeddings,
    #     distance_strategy = DistanceStrategy.COSINE
    # )

In [23]:
# vector_store.save_local("/kaggle/working/faiss_index")

## Export FAISS Index to ZIP

In [24]:
# import shutil
# shutil.make_archive("/kaggle/working/faiss_index_download", "zip", "/kaggle/working/faiss_index")

## Example query to the rag for testing

In [25]:
if RAG_KnowlegeBaseCreation:
    FAISS_INDEX_PATH = "/kaggle/input/datasets/kalyandath/faiss-index"
    vector_store = FAISS.load_local(
        folder_path = FAISS_INDEX_PATH,
        embeddings = embeddings,
        allow_dangerous_deserialization = True
    )

In [26]:
if RAG_KnowlegeBaseCreation:
    query = "What is emissivity?"
    docs_scores = vector_store.similarity_search_with_score(query, k=3)
    top_doc = docs_scores[0][0]
    
    for doc,score in docs_scores:
        print(f"Score(Distance): {score:.4f}")
        print(f"Source: {doc.metadata['source']}\n")
        print(f"Content: {doc.page_content}\n---")


---

# Model: RAG Pipeline with LLM Inference

## Load tokenizer and quantized model

In [27]:
if RAG_PIPELINE:
    RAG_INFERENCE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(RAG_INFERENCE_MODEL)
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_quant_type = "nf4",
        bnb_4bit_compute_dtype = torch.bfloat16
    )
    
    rag_inference_model = AutoModelForCausalLM.from_pretrained(
        RAG_INFERENCE_MODEL,
        quantization_config = bnb_config,
        device_map = "auto"
    )

## Define the model pipeline

In [28]:
if RAG_PIPELINE:
    text_gen_pipeline = pipeline(
        "text-generation",
        model = rag_inference_model,
        tokenizer = tokenizer,
        max_new_tokens = 128,
        temperature = 0.1,
        repetition_penalty = 1.1,
        return_full_text = False,
        pad_token_id = tokenizer.eos_token_id
    )
    
    llm = HuggingFacePipeline(pipeline = text_gen_pipeline)

## Define the cross encoder

In [29]:
if RAG_PIPELINE:
    # Vector store is not loaded yet.
    base_retriever = vector_store.as_retriever(search_kwargs={'k':15})
    ce_model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
    
    cross_encoder = HuggingFaceCrossEncoder(model_name = ce_model_name)
    compressor  = CrossEncoderReranker(model = cross_encoder, top_n = 3)
    
    compression_retriever = ContextualCompressionRetriever(
        base_compressor = compressor,
        base_retriever = base_retriever
    )

## Prepare Prompt Template and QA Chain

In [30]:
if RAG_PIPELINE:
    prompt_template='''
    You are an expert scientific multiple-choice question answering assistant.
    
    Use ONLY the provided context to answer.
    
    Rules:
    - Output ONLY one XML tag.
    - Inside the tag, write exactly THREE answer letters ranked from most likely to least likely.
    - Use only the letters A, B, C, D, and E.
    - Do not explain your reasoning.
    - Do not output any text before or after the XML tag.
    - Example xml format: <answer>A C E</answer>
    
    Context:
    {context}
    
    Question:
    {question}
    
    Answer:
    '''

In [31]:
if RAG_PIPELINE:
    PROMPT = PromptTemplate(
        template = prompt_template,
        input_variables = ["context", "question"]
    )
    
    qa_chain = RetrievalQA.from_chain_type(
        llm = llm,
        chain_type = "stuff",
        retriever = compression_retriever,
        return_source_documents = False, # True,
        chain_type_kwargs = {"prompt": PROMPT}
    )

## Answer parser

In [32]:
if RAG_PIPELINE:
    def parse_answer_option(response):
        match = re.search(r'<answer>\s*(.*?)\s*</answer>', response, re.IGNORECASE)
    
        extracted_options = []
        if match:
            tag_content = match.group(1).upper()
            extracted_options = re.findall(r'[A-E]', tag_content)
        else:
            all_letters = re.findall(r'\b([A-E])\b', response)
            extracted_options = all_letters[-3:] if all_letters else []
    
        top_3 = list(dict.fromkeys(extracted_options))
        default_options = ["A", "B", "C", "D", "E"]
    
        for letter in default_options:
            if len(top_3) >=3 :
                break
            if letter not in top_3:
                top_3.append(letter)
    
        return " ".join(top_3[:3])

In [33]:
if RAG_PIPELINE:
    parse_answer_option("<answer>C C C A E</answer> Dasfdsadf.safasdfasdf")

## Test the RAG Pipeline

In [34]:
if RAG_PIPELINE:
    def query_rag(question):
        response = qa_chain.invoke({"query": question})
        print("---------------Source-----------------------")
        # set return_source_documents arg to True first before uncomment below.
        # for doc in response['source_documents']:
        #     print(f"= {doc.page_content}")
        # print("-" * 50)
        return response["result"]

In [35]:
if RAG_PIPELINE:
    max_length = len(train_df)
    index = random.randint(0, max_length)
    
    prompt = str(train_df.iloc[index]["prompt"])
    options = ""
    
    for item in ["A", "B", "C", "D", "E"]:
        options += f"{item}) {str(train_df.iloc[index][item])}\n"
    print(prompt)
    print(options)
    
    print(f"Answer (Truth Value): {train_df.iloc[index]["answer"]}\n\n")
    
    query = prompt + '\n\n' + options
    
    print(query_rag(query))

## Prediction using RAG

In [37]:
if RAG_PIPELINE:
    predictions = []
    queries = []
    
    for index, row in test_df.iterrows():
        question = row["prompt"]
        options_text = (
            f"A. {row['A']}\n"
            f"B. {row['B']}\n"
            f"C. {row['C']}\n"
            f"D. {row['D']}\n"
            f"E. {row['E']}"
        )
        queries.append({
            "query": question + "\n\n" + options_text
        })
    
    batch_size = 10
    responses = []
    
    for i in tqdm(range(0, len(queries), batch_size), desc="Running QA Chain"):
        chunk = queries[i:i + batch_size]
        chunk_responses = qa_chain.batch(chunk)
        responses.extend(chunk_responses)
    
    for row, response in zip(test_df.itertuples(), responses):
        predictions.append({
            "id": row.id,
            "prediction": parse_answer_option(response["result"])
        })

## Submission csv creation

In [38]:
if RAG_PIPELINE:
    submission_df = pd.DataFrame(predictions)
    submission_df.to_csv("submission.csv", index = False)

----

In [39]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretClient()
# secret = user_secrets.get_secret("WANDB_API_KEY")

In [40]:
# wandb.login(api_key = secret)

# Model: Bidirectional LSTM with Attention.

## Set Seed for experiments

In [41]:
if LSTM_MODEL:
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
      torch.cuda.manual_seed(SEED)

## Setup and Config

In [42]:
if LSTM_MODEL:
    WANDB_PROJECT = "21f2000763-t22026"
    TOKENIZER = "google-bert/bert-base-uncased"
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)
    
    CURRENT_RUN_NAME = "bilstm_1_01_baseline"
    CONFIG = {
        "embed_dim": 128,
        "hidden_dim": 128,
        "dropout": 0.3,
        "learning_rate": 1e-3,
        "batch_size": 16,
        "epochs": 5
    }
    
    # CURRENT_RUN_NAME = "bilstm_02_high_dropout"
    # CONFIG = {
    #     "embed_dim": 128,
    #     "hidden_dim": 128,
    #     "dropout": 0.5,
    #     "learning_rate": 1e-3,
    #     "batch_size": 16,
    #     "epochs": 5
    # }
    
    # CURRENT_RUN_NAME = "bilstm_03_increase_dim"
    # CONFIG = {
    #     "embed_dim": 256,      # Doubled
    #     "hidden_dim": 256,     # Doubled
    #     "dropout": 0.5,
    #     "learning_rate": 1e-3,
    #     "batch_size": 16,
    #     "epochs": 5
    # }
    
    # CURRENT_RUN_NAME = "bilstm_04_lr"
    # CONFIG = {
    #     "embed_dim": 256,
    #     "hidden_dim": 256,
    #     "dropout": 0.5,
    #     "learning_rate": 5e-4, # in half
    #     "batch_size": 16,
    #     "epochs": 5
    # }


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Dataset creation

In [43]:
if LSTM_MODEL:
    label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
    train_df["labels"] = train_df["answer"].map(label_map)
    
    train_dataset = Dataset.from_pandas(train_df)
    test_dataset  = Dataset.from_pandas(test_df)

## Tokenizer Function

In [44]:
if LSTM_MODEL:
    def tokenize(batch):
      texts = [
        f"{p} [SEP] A: {a} B: {b} C: {c} D: {d} E: {e}"
        for p, a, b, c, d, e in zip(
            batch["prompt"],
            batch["A"],
            batch["B"],
            batch["C"],
            batch["D"],
            batch["E"],
        )
      ]
    
      tokenized = tokenizer(texts, padding="max_length", truncation = True, max_length=256)
    
      if 'labels' in batch:
        tokenized['labels'] = batch["labels"]
    
      return tokenized

In [45]:
if LSTM_MODEL:
    train_dataset = train_dataset.map(tokenize, batched = True)
    test_dataset  = test_dataset.map(tokenize, batched = True)
    
    train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

Map:   0%|          | 0/1758 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## DataLoader

In [46]:
if LSTM_MODEL:
    split_dataset = train_dataset.train_test_split(test_size = 0.2, seed = SEED)
    train_loader  = DataLoader(split_dataset["train"], batch_size = CONFIG["batch_size"], shuffle = True)
    val_loader    = DataLoader(split_dataset["test"], batch_size = CONFIG["batch_size"])
    test_loader   = DataLoader(test_dataset, batch_size = CONFIG["batch_size"])

## Model

In [47]:
if LSTM_MODEL:
    class BiLSTMAttention(nn.Module):
      def __init__(self, vocab_size):
        super().__init__()
    
        self.embedding = nn.Embedding(
            vocab_size,
            CONFIG["embed_dim"],
            padding_idx = 0
        )
    
        self.lstm = nn.LSTM(
            CONFIG["embed_dim"],
            CONFIG["hidden_dim"],
            batch_first = True,
            bidirectional = True
        )
    
        self.attention = nn.Linear(CONFIG["hidden_dim"] * 2 ,1)
        self.dropout   = nn.Dropout(CONFIG["dropout"])
        self.fc        = nn.Linear(CONFIG["hidden_dim"] * 2 ,5)
    
      def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids)
        x, _ = self.lstm(x)
    
        scores = self.attention(x).squeeze(-1)
    
        scores = scores.masked_fill(attention_mask == 0, -1e9)
    
        weights = torch.softmax(scores, dim=1)
    
        # Apply weights to LSTM outputs
        context = torch.sum(
            x * weights.unsqueeze(-1),
            dim = 1
        )
    
        context = self.dropout(context)
        return self.fc(context)

## MAP 3 metric function

In [48]:
if LSTM_MODEL:
    def map3_score(predictions, labels):
      score = 0.0
      for i in range(len(labels)):
        pred = predictions[i]
        label = labels[i]
    
        if label in pred:
          rank = pred.index(label)
          score += 1.0/ (rank + 1)
    
      return score / len(labels)

## Training and Validation

In [49]:
if LSTM_MODEL:
    # wandb.init(project=WANDB_PROJECT, name = CURRENT_RUN_NAME, config = CONFIG)
    model = BiLSTMAttention(tokenizer.vocab_size).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr = CONFIG["learning_rate"])
    criterion = nn.CrossEntropyLoss()
    
    print(f"Training on {DEVICE}....")
    
    for epoch in range(CONFIG["epochs"]):
      model.train()
      running_loss = 0
      for batch in train_loader:
        # Set to zero
        optimizer.zero_grad()
    
        # forward pass
        logits = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
        loss = criterion(logits, batch["labels"].to(DEVICE))
    
        # back prop
        loss.backward()
    
        # update
        optimizer.step()
    
        running_loss += loss.item()
    
      train_loss = running_loss/len(train_loader)
    
      model.eval()
      val_loss = 0
      val_correct = 0
      val_total = 0
    
      all_labels = []
      all_top3 = []
    
      val_correct, val_total = 0, 0
      with torch.no_grad():
        for batch in val_loader:
          logits = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
    
          loss = criterion(logits, batch["labels"].to(DEVICE))
          val_loss += loss.item()
          probs = torch.softmax(logits, dim=1)
          top3 = torch.argsort(
              probs,
              descending=True
          )[:, :3]
          preds = top3[:, 0]
          labels = batch["labels"].to(DEVICE)
          val_correct += (preds == labels).sum().item()
          val_total += labels.size(0)
    
    
          all_labels.extend(labels.cpu().tolist())
          all_top3.extend(top3.cpu().tolist())
    
      val_loss = val_loss / len(val_loader)
      val_map3 = map3_score(all_top3, all_labels)
      val_acc = val_correct/val_total
      print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val MAP@3: {val_map3:.4f}"
      )
      # wandb.log({
      #     'epoch': epoch + 1,
      #     "train_loss": train_loss,
      #     "val_loss": val_loss,
      #     "val_accuracy": val_acc,
      #     "val_map3": val_map3,
      #     "learning_rate": optimizer.param_groups[0]["lr"]
      # })
    
    # wandb.finish()

Training on cpu....
Epoch 1 | Train Loss: 1.4516 | Val Loss: 1.1136 | Val Acc: 0.5881 | Val MAP@3: 0.7197
Epoch 2 | Train Loss: 0.6076 | Val Loss: 0.3530 | Val Acc: 0.8920 | Val MAP@3: 0.9408
Epoch 3 | Train Loss: 0.1304 | Val Loss: 0.0298 | Val Acc: 1.0000 | Val MAP@3: 1.0000
Epoch 4 | Train Loss: 0.0162 | Val Loss: 0.0131 | Val Acc: 0.9972 | Val MAP@3: 0.9986
Epoch 5 | Train Loss: 0.0899 | Val Loss: 0.0931 | Val Acc: 1.0000 | Val MAP@3: 1.0000


## Inference

In [50]:
if LSTM_MODEL:
    print("Running Inference ....")
    
    model.eval()
    predictions = []
    
    index_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
    
    with torch.no_grad():
      for batch in test_loader:
        logits = model(batch["input_ids"].to(DEVICE), batch['attention_mask'].to(DEVICE))
        probs = torch.nn.functional.softmax(logits, dim = 1)
        # Top 3 preds
        top3 = torch.argsort(probs, dim = 1, descending = True)[:, :3].cpu().numpy()
        for row in top3:
          predictions.append(" ".join([index_to_letter[idx] for idx in row]))

Running Inference ....


## Submission csv creation

In [51]:
if LSTM_MODEL:
    submission_df = pd.DataFrame({
        "id": test_df["id"],
        "prediction": predictions
    })
    
    submission_df.to_csv("submission.csv", index = False)
    submission_df.head()

---

# Model: Pretrained Deberta Model with training.

## Training: Wandb Init

In [52]:
if DEBERTA_TRAINING:
    WANDB_RUN_NAME = "deberta-v3-finetune-run1"
    wandb.init(
        project="21f2000763-t22026",
        name= WANDB_RUN_NAME,
        tags = ["deberta", "transformer"]
    )

## Training: Load Tokenizer

In [53]:
if DEBERTA_TRAINING:
    tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
    test_sentence = "What is emissivity?"
    inputs = tokenizer(test_sentence)
    print(inputs)

## Training: Create Train dataset

In [54]:
if DEBERTA_TRAINING:
    train_dataset = Dataset.from_pandas(train_df)
    print(train_dataset)

## Training: Create Train Test Split

In [55]:
if DEBERTA_TRAINING:
    split_dataset = train_dataset.train_test_split(test_size = 0.2)
    split_dataset

## Training: Tokenization Function

In [56]:
if DEBERTA_INFERENCE or DEBERTA_TRAINING:
    ## tokenization
    def preprocess(data):
        output = {
            "input_ids": [],
            "attention_mask": [],
            "labels": []
        }
    
        label_map = {'A':0, 'B':1,'C':2, 'D':3, 'E':4}
    
        prompts = []
        options = []
        answer = []
    
        for i in range(len(data["prompt"])):
            prompts += [data["prompt"][i]] * 5
            options += [data[x][i] for x in label_map.keys()]
            answer.append(label_map[data["answer"][i]])
    
        tokenized_input = tokenizer(prompts, options, truncation = True, max_length = 256)
        
        for i in range(0, len(tokenized_input["input_ids"]),5):
            output["input_ids"].append(tokenized_input["input_ids"][i:i+5])
            output["attention_mask"].append(tokenized_input["attention_mask"][i:i+5])
            output["labels"].append(answer[i//5])
    
        print(output["input_ids"][0])
        return output

## Training: Apply Preprocess function

In [57]:
if DEBERTA_TRAINING:
    tokenized_dataset = split_dataset.map(preprocess, batched = True)
    print(tokenized_dataset)

## Training: Data Collator for multiple choice

In [58]:
if DEBERTA_TRAINING or DEBERTA_INFERENCE:
    @dataclass
    class DataCollatorForMultipleChoice:
        tokenizer: any
    
        def __call__(self, features):
            flat_input = {
                "input_ids": [],
                "attention_mask": []
            }
            labels = []
        
            for item in features:
                flat_input["input_ids"] += item["input_ids"]
                flat_input["attention_mask"] += item["attention_mask"]
                labels.append(item["labels"])
        
            batch = self.tokenizer.pad(
                flat_input,
                return_tensors = "pt"
            )
        
            batch["input_ids"] = batch["input_ids"].view(len(features), 5, -1)
            batch["attention_mask"] = batch["attention_mask"].view(len(features), 5, -1)
            batch["labels"] = torch.tensor(labels)
        
            return batch
    
    data_collator = DataCollatorForMultipleChoice(tokenizer = tokenizer)

## Training : Load Model

In [59]:
if DEBERTA_TRAINING:
    model = AutoModelForMultipleChoice.from_pretrained("microsoft/deberta-v3-base", torch_dtype=torch.float32)

## Training: Set up Training Arguments

In [60]:
if DEBERTA_TRAINING:
    training_args = TrainingArguments(
        output_dir = "./results",
        save_strategy = "epoch",
        learning_rate = 1e-5,
        per_device_train_batch_size = 4,
        num_train_epochs = 3,
        weight_decay = 0.01,
        fp16 = False,
        max_grad_norm = 1.0,
        logging_steps = 50,
        # report_to = "wandb",
        # run_name = WANDB_RUN_NAME
    )

## Training: Initialize the trainer

In [61]:
if DEBERTA_TRAINING:
    trainer = Trainer(
        model = model,
        args = training_args,
        train_dataset = tokenized_dataset["train"],
        eval_dataset  = tokenized_dataset["test"],
        data_collator = data_collator
    )

## Training: Model Train

In [62]:
if DEBERTA_TRAINING:
    trainer.train()
    # wandb.finish()

## Training: Save Model for Inference

In [63]:
if DEBERTA_TRAINING:
    trainer.save_model("./my_deberta_model")
    tokenizer.save_pretrained("./my_deberta_model")

## Inference: Load tokenizer and model

In [64]:
if DEBERTA_INFERENCE:
    DEBERTA_MODEL = "/kaggle/input/datasets/kalyandath/deberta-pretrained/"
    tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL)
    model = AutoModelForMultipleChoice.from_pretrained(DEBERTA_MODEL)

## Inference: Dataset creation

In [65]:
if DEBERTA_INFERENCE:
    test_df["answer"] = "A"
    test_dataset = Dataset.from_pandas(test_df)
    tokenized_test = test_dataset.map(preprocess, batched=True)

## Inference: Define trainer and run prediction.

In [67]:
if DEBERTA_INFERENCE:
    data_collator = DataCollatorForMultipleChoice(tokenizer = tokenizer)
    inference_trainer = Trainer(
        model = model,
        data_collator = data_collator
    )
    
    print("Running Predictions")
    predictions = inference_trainer.predict(tokenized_test)
    logits = predictions.predictions

## Inference: Extract options from predictions.

In [68]:
if DEBERTA_INFERENCE:
    top_3_indices = np.argsort(logits, axis=1)[:,-1:-4:-1]
    index_to_letter = {0:"A", 1:"B", 2:"C", 3:"D", 4: "E"}
    predicted_strings = []
    
    for row in top_3_indices:
        letters = [index_to_letter[idx] for idx in row]
        predicted_strings.append(" ".join(letters))

## Inference: Submission csv creation

In [69]:
if DEBERTA_INFERENCE:
    # Submission for pretrained model
    submission_df = pd.DataFrame({
        "id": test_df["id"],
        "prediction": predicted_strings
    })
    submission_df.to_csv("submission.csv", index= False)
    print(submission_df.head())